# Lab 4 — Data Visualization with Matplotlib and Seaborn

**DS331 · Introduction to Data Science**

A good chart communicates a finding faster than any table. This is the core skill for your report, so we go step by step: first the *anatomy* of a matplotlib figure, then the gallery of plot types — each introduced with **the question it answers**.

1. [Anatomy of a figure — matplotlib from zero](#1)
2. [The basic matplotlib plots](#2)
3. [Multiple plots in one figure — subplots](#3)
4. [The pandas `.plot()` shortcut](#4)
5. [Seaborn — statistical plots made easy](#5)
6. [Distributions: histogram, KDE, box, violin](#6)
7. [Relationships: scatter, regression, line](#7)
8. [Categorical comparisons: count, bar, grouped box](#8)
9. [Correlation heatmaps](#9)
10. [Multi-panel figures: pairplot and catplot](#10)
11. [Styling, palettes, and saving figures](#11)
12. [Which plot for which question? — cheat sheet](#12)
13. [Try it yourself ✏️](#13)

**Datasets:** *tips* (restaurant bills) and *penguins* — both clean and intuitive, so we can focus purely on plotting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

tips = sns.load_dataset("tips")
penguins = sns.load_dataset("penguins").dropna()   # cleaned, as we learned in Lab 2

tips.head(3)

<a id="1"></a>
## 1. Anatomy of a figure — matplotlib from zero

**Matplotlib** is the foundation library — seaborn and pandas plotting are built on top of it. Understanding its parts lets you customize *any* plot later.

A plot lives inside two objects:
- the **Figure** — the whole canvas (can hold several plots);
- the **Axes** — one plot: its x-axis, y-axis, and the data drawn inside.

The recommended pattern is to create them explicitly:

In [ ]:
x = np.linspace(0, 10, 50)          # 50 evenly spaced numbers from 0 to 10
y = np.sin(x)

fig, ax = plt.subplots()            # 1. create Figure + Axes
ax.plot(x, y)                       # 2. draw on the Axes
plt.show()                          # 3. display

That works, but a bare plot is unreadable to anyone else. Now the same plot **fully labeled** — watch how each line adds one element:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))                    # figsize = (width, height) in inches

ax.plot(x, np.sin(x), color="tab:blue", linewidth=2,
        linestyle="-", marker="o", markersize=4, label="sin(x)")
ax.plot(x, np.cos(x), color="tab:orange", linewidth=2,
        linestyle="--", label="cos(x)")

ax.set_title("Sine and cosine")                           # the headline of the plot
ax.set_xlabel("x")                                        # ALWAYS label your axes
ax.set_ylabel("value")
ax.legend()                                               # shows the 'label' of each line
ax.grid(True, alpha=0.3)                                  # subtle background grid

plt.show()

**The golden rule for your report:** every figure needs a *title*, *axis labels*, and (when there are several series) a *legend*. A plot without these is incomplete.

> ⚠️ **Common mistake:** you may see tutorials using `plt.title(...)`, `plt.xlabel(...)` directly. That works for a single plot, but breaks down with multiple subplots. The `ax.set_...` style always works — we use it throughout.

<a id="2"></a>
## 2. The basic matplotlib plots

Four workhorses, each in its simplest form. (Seaborn will make fancier versions later — but you should know what is underneath.)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

# --- line plot: trends over an ordered axis (usually time) ---
days = np.arange(1, 31)
sales = 100 + days * 2 + np.random.RandomState(0).normal(0, 8, 30)
axes[0, 0].plot(days, sales)
axes[0, 0].set_title("Line — trend over time")
axes[0, 0].set_xlabel("day"); axes[0, 0].set_ylabel("sales")

# --- bar chart: compare quantities across categories ---
species_counts = penguins["species"].value_counts()
axes[0, 1].bar(species_counts.index, species_counts.values, color=["#4C72B0", "#DD8452", "#55A868"])
axes[0, 1].set_title("Bar — counts per category")
axes[0, 1].set_xlabel("species"); axes[0, 1].set_ylabel("count")

# --- histogram: the distribution of one numeric column ---
axes[1, 0].hist(penguins["body_mass_g"], bins=25, edgecolor="white")
axes[1, 0].set_title("Histogram — distribution of body mass")
axes[1, 0].set_xlabel("body mass (g)"); axes[1, 0].set_ylabel("frequency")

# --- scatter plot: relationship between two numeric columns ---
axes[1, 1].scatter(penguins["flipper_length_mm"], penguins["body_mass_g"], alpha=0.6, s=20)
axes[1, 1].set_title("Scatter — two numeric variables")
axes[1, 1].set_xlabel("flipper length (mm)"); axes[1, 1].set_ylabel("body mass (g)")

fig.tight_layout()
plt.show()

A note on **pie charts**: they exist (`ax.pie(...)`) but are hard to read with more than 3–4 slices — humans compare lengths better than angles. A bar chart is almost always the better choice; if you do use a pie, keep it to a handful of categories.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(species_counts.values, labels=species_counts.index, autopct="%1.0f%%", startangle=90)
ax.set_title("Share of each species (pie — use sparingly!)")
plt.show()

<a id="3"></a>
## 3. Multiple plots in one figure — subplots

You already saw it above: `plt.subplots(rows, cols)` returns a **grid of Axes**, and you draw on each one. This is how you build comparison figures.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)   # sharey: same y-scale -> fair comparison

for ax, species in zip(axes, ["Adelie", "Chinstrap", "Gentoo"]):
    subset = penguins[penguins["species"] == species]
    ax.hist(subset["body_mass_g"], bins=15, edgecolor="white")
    ax.set_title(species)
    ax.set_xlabel("body mass (g)")

axes[0].set_ylabel("frequency")
fig.suptitle("Body mass distribution by species", y=1.04, fontsize=13)
fig.tight_layout()
plt.show()

Three things to remember:
- with a 1-row grid, `axes` is a 1-D array → `axes[0]`; with 2-D grids → `axes[row, col]`;
- `sharey=True` (or `sharex`) keeps scales identical so panels are comparable;
- `fig.tight_layout()` fixes overlapping labels — add it whenever things look cramped.

<a id="4"></a>
## 4. The pandas `.plot()` shortcut

Every DataFrame/Series has a built-in `.plot()` that calls matplotlib for you — unbeatable for quick checks while exploring:

In [ ]:
penguins.groupby("species")["body_mass_g"].mean().plot(
    kind="bar", figsize=(6, 3.5), title="Average body mass by species", ylabel="body mass (g)"
)
plt.show()

`kind=` accepts `"line"`, `"bar"`, `"barh"`, `"hist"`, `"box"`, `"scatter"`, `"pie"`… Perfect for exploration; for the polished figures in your report, seaborn (next) usually looks better with less work.

<a id="5"></a>
## 5. Seaborn — statistical plots made easy

**Seaborn** is built on matplotlib and adds two superpowers:

1. it understands **DataFrames** — you pass `data=df` and refer to columns by name;
2. the magic **`hue=`** argument — color the plot by a categorical column, instantly splitting any plot into groups.

The basic recipe for almost every seaborn call:

```python
sns.someplot(data=df, x="col1", y="col2", hue="group_col")
```

Let's set a clean default theme for everything that follows:

In [ ]:
sns.set_theme(style="whitegrid")

<a id="6"></a>
## 6. Distributions — *"how is this variable spread out?"*

The first question you ask about every numeric column. Four views of the same idea:

### 6.1 `histplot` — the histogram

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=tips, x="total_bill", bins=25, ax=ax)
ax.set_title("Distribution of total bill")
plt.show()

Read a histogram by asking: where is the bulk? is it symmetric or **skewed**? are there several bumps (groups mixed together)? are there outliers far out?

Here: most bills are 10–20 dollars, with a right skew (a long tail of expensive bills).

> ⚠️ The `bins` choice changes the picture! Too few bins hide structure, too many show noise. Try 2–3 values before settling.

### 6.2 `kdeplot` — a smooth version, and `hue` to compare groups

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(data=tips, x="total_bill", hue="time", fill=True, alpha=0.4, ax=ax)
ax.set_title("Bill distribution: lunch vs dinner")
plt.show()

One `hue="time"` and we instantly compare lunch vs dinner: dinner bills are larger and more spread out. This is the seaborn way — *splitting by a category costs one argument*.

### 6.3 `boxplot` — distributions side by side

The box shows the quartiles (middle 50%), the line is the median, whiskers extend 1.5×IQR, and dots beyond are outliers (exactly the rule from Lab 2!).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=tips, x="day", y="total_bill", hue="day", palette="pastel", legend=False, ax=ax)
ax.set_title("Total bill by day of week")
plt.show()

### 6.4 `violinplot` — box + shape

A violin is a boxplot with the full distribution shape drawn around it — it reveals things a box hides (like two bumps):

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.violinplot(data=tips, x="day", y="total_bill", hue="sex", split=True, inner="quart", ax=ax)
ax.set_title("Bill distribution by day and payer sex")
plt.show()

<a id="7"></a>
## 7. Relationships — *"how do two numeric variables move together?"*

### 7.1 `scatterplot` — and encoding extra variables

A scatter plot can carry up to **four** variables at once: x, y, `hue` (color), and `size`:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=penguins, x="flipper_length_mm", y="body_mass_g",
                hue="species", style="species", s=60, alpha=0.8, ax=ax)
ax.set_title("Flipper length vs body mass")
plt.show()

Two findings in one plot: the variables are strongly positively related, *and* the species form visible clusters (Gentoo top-right). Always ask your scatter plots both questions: overall trend, and group structure.

### 7.2 `regplot` — scatter + fitted trend line

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.regplot(data=tips, x="total_bill", y="tip",
            scatter_kws={"alpha": 0.5}, line_kws={"color": "red"}, ax=ax)
ax.set_title("Do bigger bills bring bigger tips?")
plt.show()

> ⚠️ A trend line is *descriptive*, not proof of causation. "Bills and tips rise together" ≠ "big bills *cause* big tips".

### 7.3 `lineplot` — ordered data (time, sequences)

For trends over an ordered axis. When several rows share an x value, seaborn draws the **mean line and a confidence band** automatically:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(data=tips, x="size", y="tip", marker="o", ax=ax)
ax.set_title("Average tip by party size (band = 95% confidence interval)")
plt.show()

<a id="8"></a>
## 8. Categorical comparisons — *"how do groups differ?"*

### 8.1 `countplot` — frequency of each category (the categorical histogram)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=tips, x="day", hue="time", ax=ax)
ax.set_title("Number of parties by day and time")
plt.show()

### 8.2 `barplot` — a *statistic* per category (mean by default)

Note the difference: `countplot` counts rows; `barplot` aggregates a numeric column. The black whiskers are confidence intervals — wide whiskers mean few observations, so be careful comparing those bars.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=tips, x="day", y="total_bill", estimator="mean", ax=ax)
ax.set_title("Average bill by day")
plt.show()

### 8.3 Grouped boxplots — the most informative group comparison

Bars show only the mean; boxes show the whole distribution. For your report, grouped boxplots are usually the strongest "compare groups" figure:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=penguins, x="species", y="bill_length_mm", hue="sex", ax=ax)
ax.set_title("Bill length by species and sex")
plt.show()

<a id="9"></a>
## 9. Correlation heatmaps — *"which variables move together?"*

The **correlation coefficient** ranges from −1 (perfect opposite movement) through 0 (no linear relationship) to +1 (perfect joint movement). `df.corr()` computes it for every pair of numeric columns, and a heatmap makes the matrix readable:

In [ ]:
corr = penguins.select_dtypes("number").corr()   # numeric columns only!

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation between penguin measurements")
plt.show()

How to read it: flipper length and body mass correlate at **0.87** — big penguins have big flippers (we saw exactly this in the scatter plot). Bill depth correlates *negatively* with the others — surprising! (It is a species effect: the heaviest species happens to have shallow bills. Another reminder that correlation has to be interpreted, not just reported.)

Key arguments: `annot=True` writes the numbers, `cmap="coolwarm"` makes negative blue / positive red, `vmin/vmax=-1, 1` fixes the scale honestly.

> ⚠️ **Common mistake:** `df.corr()` on a DataFrame with text columns raises an error — select numeric columns first with `df.select_dtypes("number")`.

<a id="10"></a>
## 10. Multi-panel figures — `pairplot` and `catplot`

### 10.1 `pairplot` — every numeric pair at once

One line gives scatter plots of every variable pair, with distributions on the diagonal. The classic *first look* at a new dataset:

In [ ]:
sns.pairplot(penguins, hue="species", corner=True, height=2.0)
plt.show()

(For datasets with many columns, pass a subset: `sns.pairplot(df[["a", "b", "c", "target"]], hue="target")` — a 30-column pairplot is unreadable.)

### 10.2 `catplot` — a grid of plots split by category

`col=` creates one panel per value of a column. Great for "same plot, per group" comparisons:

In [ ]:
sns.catplot(data=tips, x="day", y="total_bill", hue="sex",
            col="time", kind="box", height=4, aspect=1.0)
plt.show()

`kind=` accepts `"box"`, `"violin"`, `"bar"`, `"count"`, `"strip"`… — one interface, many plots. (Its scatter/line sibling is `sns.relplot`.)

<a id="11"></a>
## 11. Styling, palettes, and saving

### 11.1 Themes and contexts

```python
sns.set_theme(style="whitegrid")   # styles: whitegrid, darkgrid, white, dark, ticks
sns.set_context("notebook")        # contexts: paper, notebook, talk, poster (scales all fonts)
```

Use `talk` when preparing slides — everything becomes bigger and readable from the back row.

### 11.2 Color palettes

Color should *mean* something. Three families, three uses:
- **Qualitative** (`"Set2"`, `"tab10"`) — unordered categories (species, city);
- **Sequential** (`"viridis"`, `"Blues"`) — ordered/low-to-high values;
- **Diverging** (`"coolwarm"`, `"RdBu"`) — values around a meaningful center (correlations around 0).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

sns.countplot(data=penguins, x="species", hue="species", palette="Set2", legend=False, ax=axes[0])
axes[0].set_title('qualitative — "Set2"')

sns.histplot(data=penguins, x="body_mass_g", hue="species",
             multiple="stack", palette="viridis", ax=axes[1])
axes[1].set_title('sequential — "viridis"')

sns.heatmap(penguins.select_dtypes("number").corr(), cmap="coolwarm",
            vmin=-1, vmax=1, cbar=False, ax=axes[2])
axes[2].set_title('diverging — "coolwarm"')

fig.tight_layout()
plt.show()

### 11.3 Saving figures for your report

`savefig` must be called **before** `plt.show()` (after `show()`, the figure is cleared):

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=tips, x="day", y="total_bill", ax=ax)
ax.set_title("Total bill by day")

fig.savefig("bill_by_day.png", dpi=150, bbox_inches="tight")   # dpi=150+ for crisp images
plt.show()

print("saved bill_by_day.png — in Colab, find it in the file panel on the left")

<a id="12"></a>
## 12. Which plot for which question? — cheat sheet

| Your question | Plot | Function |
|---|---|---|
| How is one numeric variable distributed? | histogram / KDE | `sns.histplot`, `sns.kdeplot` |
| Does it differ across groups? | grouped box / violin | `sns.boxplot`, `sns.violinplot` |
| How often does each category occur? | count plot | `sns.countplot` |
| What is the average X per group? | bar plot | `sns.barplot` |
| How do two numeric variables relate? | scatter (+ trend) | `sns.scatterplot`, `sns.regplot` |
| How does something change over time/order? | line plot | `sns.lineplot` |
| Which variables correlate? | heatmap of `corr()` | `sns.heatmap` |
| First overview of everything? | pair plot | `sns.pairplot` |
| Same plot for each subgroup? | panel grid | `sns.catplot` (`col=`) |

And the quality checklist for **every** figure in your report:

- [ ] title states what the plot shows (even better: states the *finding*)
- [ ] both axes labeled, with units
- [ ] legend present if colors/styles encode something
- [ ] readable font sizes; no overlapping labels
- [ ] one sentence of interpretation *below the figure* — a plot nobody explains is a plot nobody needed

<a id="13"></a>
## 13. Try it yourself ✏️

All on the `penguins` dataset. For each plot, include title + axis labels, and write one sentence on what it shows.

**Exercise 1.** A histogram of `bill_length_mm` with a sensible number of bins. Is the distribution skewed? Could there be more than one bump — and why might that be?

**Exercise 2.** A scatter plot of `bill_length_mm` vs `bill_depth_mm`, colored by species. Do clusters appear?

**Exercise 3.** Grouped boxplots of `flipper_length_mm` per island. Which island hosts the biggest penguins? (Bonus: which species lives there? Check with `value_counts`.)

**Exercise 4.** A figure with **two subplots side by side**: left, a countplot of species; right, a barplot of mean `body_mass_g` per species. Use `plt.subplots(1, 2, ...)`.

**Exercise 5.** Save your favorite of the four as a PNG at `dpi=150` and download it from the Colab file panel.

In [ ]:
# Exercise 1

In [ ]:
# Exercise 2

In [ ]:
# Exercise 3

In [ ]:
# Exercise 4

In [ ]:
# Exercise 5

---
**Next lab:** static plots are the backbone of a report — but interactive plots let you *explore*. **Lab 5 — Interactive Plots with Plotly + a complete EDA walkthrough** ties the whole course together.